# Inserting Multiple Documents with `insertMany()`

`Model.insertMany()` inserts an array of documents in a **single database round-trip**, after validating each one against your schema. Compare this with calling `save()` in a loop, which costs one round-trip per document.

```
Model.insertMany(docs, [options])  →  Promise<Document[]>
```

## Basic example

```javascript
const mongoose = require('mongoose');

// 1. Define a schema and model
const userSchema = new mongoose.Schema({
  name: { type: String, required: true },
  age: Number,
  role: String
});

const User = mongoose.model('User', userSchema);

// 2. Perform the insertMany operation
async function insertUsers() {
  const usersArray = [
    { name: 'Alice', age: 25, role: 'Admin' },
    { name: 'Bob', age: 30, role: 'User' },
    { name: 'Charlie', age: 35, role: 'Moderator' }
  ];

  try {
    const docs = await User.insertMany(usersArray);
    console.log('Successfully inserted documents:', docs);
  } catch (error) {
    console.error('Error inserting documents:', error);
  }
}
```

Typical seed-script usage:

```javascript
Movie.insertMany([
  { title: 'Amelie', year: 2001, score: 8.3, rating: 'R' },
  { title: 'Alien', year: 1979, score: 8.1, rating: 'R' },
  { title: 'The Iron Giant', year: 1999, score: 7.5, rating: 'PG' },
  { title: 'Stand By Me', year: 1986, score: 8.6, rating: 'R' },
  { title: 'Moonrise Kingdom', year: 2012, score: 7.3, rating: 'PG-13' }
])
  .then(data => console.log(data))
  .catch(err => console.log(err));
```

Standalone seed scripts hold the connection open after inserting — end with `mongoose.disconnect()` or the process hangs.

## Options

Pass an optional object as the second parameter: `Model.insertMany(docs, options)`.

### 1. Ordered vs unordered (`ordered`)

- **`ordered: true` (default)** — documents are inserted sequentially. On the first error (validation failure, duplicate key), execution stops and the remaining documents are skipped. Documents already inserted stay inserted; there's no rollback outside a transaction.
- **`ordered: false`** — the server may insert in parallel and continues past failures, inserting every document that's valid.

```javascript
// Continues inserting the rest even if one fails validation or throws a duplicate key error
await User.insertMany(usersArray, { ordered: false });
```

With `ordered: false`, failures don't reject the promise by default — you have to inspect the result. Opt into throwing instead:

```javascript
try {
  await User.insertMany(usersArray, { ordered: false, throwOnValidationError: true });
} catch (err) {
  console.log(err.validationErrors); // per-document ValidationErrors
  console.log(err.results);          // what did and didn't make it
}
```

### 2. Skipping document construction (`rawResult`, `lean`)

By default Mongoose returns an array of hydrated documents complete with `_id` values. Building those objects costs time on large inserts.

```javascript
// Raw driver payload: { acknowledged, insertedCount, insertedIds }
const result = await User.insertMany(usersArray, { rawResult: true });

// Plain JS objects instead of hydrated documents — still validated
const docs = await User.insertMany(usersArray, { lean: true });
```

### 3. Other useful options

| Option | Effect |
|---|---|
| `session` | Runs inside a transaction (`{ session }` from `startSession()`) |
| `limit` | Caps how many documents are processed in parallel — keeps memory bounded on huge arrays |
| `validateBeforeSave: false` | Skips schema validation entirely; only for data you already trust |
| `populate` | Populates the returned documents |

## Validation

Unlike the native MongoDB driver, Mongoose runs **full schema validation on every object in the array** before sending anything to the database. Defaults, casting, `timestamps` and setters are all applied.

The consequence: with `ordered: true`, one bad document at index 3 means documents 4+ never reach the database, while 0–2 are already committed. Validate the input yourself, use `ordered: false`, or wrap the whole thing in a transaction if partial writes are unacceptable.

## Middleware limitation

`insertMany()` does **not** trigger `pre('save')` or `post('save')` hooks.

This is the single most common source of bugs with it: password hashing, slug generation and any other logic living in a save hook is silently skipped, and you end up with plaintext passwords in the collection. If save hooks are required, use `Model.create(arrayOfDocs)` instead — it's slower (one insert per document) but runs the full document lifecycle.

`insertMany` does have its own hook, which fires once for the whole batch:

```javascript
userSchema.pre('insertMany', function (next, docs) {
  for (const doc of docs) {
    doc.createdBy = 'seed-script';
  }
  next();
});
```

Note the signature — `this` is the model, and the array of raw documents comes in as the second argument.

## Choosing between the methods

| Method | Round-trips | Validation | `save` hooks | Use when |
|---|---|---|---|---|
| `save()` in a loop | One per doc | Yes | Yes | A handful of docs needing per-doc logic |
| `create([...])` | One per doc | Yes | Yes | You need save hooks on a batch |
| `insertMany()` | One | Yes | No | Bulk inserts, seeding, imports |
| `bulkWrite()` | One | No (by default) | No | Mixed inserts/updates/deletes in one trip |
| `collection.insertMany()` | One | No | No | Raw driver escape hatch; bypasses Mongoose entirely |

## Practical notes

- **Batch size** — a single BSON command is capped at 16 MB, and the driver splits large arrays into batches automatically. Very large imports (100k+) still benefit from chunking manually so a failure doesn't cost you the whole run.
- **Duplicate keys** — surface as `MongoServerError` with `code: 11000`, not a `ValidationError`. With `ordered: false`, they appear in `err.writeErrors` rather than aborting.
- **Return order** — the returned array matches the input order, so you can zip results back to source data by index.
- **Empty array** — `insertMany([])` resolves to `[]` without hitting the database; no need to guard it.
- **Idempotency** — re-running a seed script duplicates everything unless there's a unique index or you clear the collection first. `await Movie.deleteMany({})` at the top of a seed file is the usual pattern.

## Sources

- [Model.insertMany() API](https://mongoosejs.com/docs/api/model.html)
- [Mongoose middleware](https://mongoosejs.com/docs/middleware.html)
- [MongoDB insertMany docs](https://www.mongodb.com/docs/manual/reference/method/db.collection.insertMany/)